# 05 — Land cover & soils reclassification → URH ingredients

**Goal.** Turn the two raw layers (ESA WorldCover land cover + IGAC soils) into the two inputs the **URH** needs, by
reclassifying each into a few **hydrological classes**, aligning them to the DEM grid, and crossing them — the real-data
version of notebook `02_urh`.

This is **extent-independent** prep: the reclassification schemes below carry over to whatever final domain the advisor
confirms. Only the final clipped raster changes.

> Run from the `notebooks/` folder. Requires: numpy, rasterio, geopandas.
> Heavy step (rasterising soils + resampling 10 m land cover to the DEM grid) — fine on a workstation; may be slow.

In [2]:
import os, zipfile, numpy as np, rasterio, geopandas as gpd
from rasterio.enums import Resampling
from rasterio.features import rasterize
RAW="../data/raw"; PROC="../data/processed"; os.makedirs(PROC, exist_ok=True)
DEM_TIF = os.path.join(PROC,"cop30_dem.tif")   # produced by notebook 04


## 1. Land cover → hydrological classes

ESA WorldCover has 11 codes. We collapse them into the classes that matter for the water balance / MUSLE cover factor.
Water and wetland are kept separate because they behave very differently hydrologically.

In [3]:
# reclassification: WorldCover code -> (hydro id, name)
LC_MAP = {10:(1,"Forest"), 20:(2,"Shrub"), 30:(3,"Grassland"), 40:(4,"Cropland"),
          50:(5,"Urban"),  60:(6,"Bare"),  70:(6,"Bare"),      80:(7,"Water"),
          90:(8,"Wetland"),95:(8,"Wetland"),100:(6,"Bare")}
LC_NAMES = {1:"Forest",2:"Shrub",3:"Grassland",4:"Cropland",5:"Urban",6:"Bare",7:"Water",8:"Wetland"}

# extract the WorldCover Map tiles from the zip (once)
lc_dir = os.path.join(PROC,"worldcover"); os.makedirs(lc_dir, exist_ok=True)
zf = zipfile.ZipFile(os.path.join(RAW,"landcover","worldcover_2021.zip"))
tiles=[]
for n in zf.namelist():
    if n.endswith("_Map.tif"):
        out=os.path.join(lc_dir, os.path.basename(n))
        if not os.path.exists(out): open(out,"wb").write(zf.read(n))
        tiles.append(out)
print("WorldCover tiles:", [os.path.basename(t) for t in tiles])


WorldCover tiles: ['ESA_WorldCover_10m_2021_v200_N09W075_Map.tif', 'ESA_WorldCover_10m_2021_v200_N06W075_Map.tif']


We resample the 10 m land cover onto the **DEM grid** (30 m) with *majority* resampling (correct for categorical
data), reclassify to the hydrological ids, and save an aligned raster.

In [4]:
with rasterio.open(DEM_TIF) as dem:
    dst_prof = dem.profile; dst = np.zeros((dem.height, dem.width), "uint8")
    for tp in tiles:
        with rasterio.open(tp) as src:
            tmp = np.zeros((dem.height, dem.width), "uint8")
            rasterio.warp.reproject(source=rasterio.band(src,1), destination=tmp,
                src_transform=src.transform, src_crs=src.crs,
                dst_transform=dem.transform, dst_crs=dem.crs, resampling=Resampling.mode)
            dst = np.where((dst==0)&(tmp>0), tmp, dst)     # fill from each tile
# reclassify raw WorldCover -> hydro id
lc = np.zeros_like(dst)
for raw,(hid,_) in LC_MAP.items(): lc[dst==raw]=hid
dst_prof.update(dtype="uint8", nodata=0, count=1)
with rasterio.open(os.path.join(PROC,"landcover_hydro_30m.tif"),"w",**dst_prof) as o: o.write(lc,1)
u,c = np.unique(lc[lc>0], return_counts=True)
print("Land-cover hydro classes (share):")
for k,v in sorted(zip(u,c), key=lambda x:-x[1]): print("  %-10s %5.1f%%"%(LC_NAMES[k],100*v/c.sum()))


Land-cover hydro classes (share):
  Grassland   45.1%
  Forest      32.3%
  Water       10.8%
  Wetland      9.5%
  Cropland     0.9%
  Urban        0.7%
  Shrub        0.4%
  Bare         0.3%


## 2. Soils → landscape/hydrological groups (first pass)

The merged IGAC soils (18,217 polygons) carry a `PAISAJE` (landscape) field with ~28 spellings. We harmonise them into
a handful of **landscape groups** — a defensible *first pass* that proxies hydrological behaviour (steep mountains shed
water fast; flat plains/valleys store and infiltrate it).

> **Caveat:** this is a landscape (geomorphology) proxy, not a true Hydrologic Soil Group. The proper refinement is to
> derive texture / infiltration class from the soil taxonomy fields (`TAXONOMIA`, `CARACT_SUELOS`) or from SoilGrids.
> Recorded here so it isn't forgotten.

In [5]:
soils = gpd.read_file(os.path.join(PROC,"soils_magdalena_merged_4326.gpkg"))

def group_paisaje(s):
    s = str(s).upper()
    if "MONTAÑA" in s:               return (1,"Mountain")
    if "LOMA" in s or "LOMER" in s:  return (2,"Hills")
    if "PIEDEMONTE" in s or "PIE DE MONTE" in s: return (3,"Piedmont")
    if "ALTIPLANICIE" in s:          return (4,"Plateau")
    if "PLANICIE" in s:              return (5,"Plain")
    if "VALLE" in s:                 return (6,"Valley")
    if "CUERPO" in s:                return (7,"Water")
    if "URBAN" in s:                 return (8,"Urban")
    if "ROCOSO" in s or "MINERA" in s: return (9,"Rock/Other")
    return (9,"Rock/Other")

SOIL_NAMES = {1:"Mountain",2:"Hills",3:"Piedmont",4:"Plateau",5:"Plain",6:"Valley",7:"Water",8:"Urban",9:"Rock/Other"}
soils["SOIL_ID"] = soils["PAISAJE"].map(lambda s: group_paisaje(s)[0])
print("Soil landscape groups (polygon counts):")
print(soils["SOIL_ID"].map(SOIL_NAMES).value_counts().to_string())


DataSourceError: sqlite3_prepare_v2(SELECT COUNT(*) FROM sqlite_master WHERE name IN ('gpkg_metadata', 'gpkg_metadata_reference') AND type IN ('table', 'view')) failed: attempt to write a readonly database; attempt to write a readonly database

Rasterise the soil groups onto the DEM grid so soils and land cover share the exact same cells (the alignment
prerequisite from notebook 02 — "everything on the DEM grid").

In [ ]:
with rasterio.open(DEM_TIF) as dem:
    shapes = ((geom, sid) for geom, sid in zip(soils.geometry, soils["SOIL_ID"]))
    soil_ras = rasterize(shapes, out_shape=(dem.height,dem.width), transform=dem.transform,
                         fill=0, dtype="uint8")
    sp = dem.profile; sp.update(dtype="uint8", nodata=0, count=1)
    with rasterio.open(os.path.join(PROC,"soils_hydro_30m.tif"),"w",**sp) as o: o.write(soil_ras,1)
print("soils rasterised to the DEM grid ->", "soils_hydro_30m.tif")


## 3. Cross soil × land cover → URH

Exactly notebook 02's index formula, now on the real grid:

$$\text{URH} = (\text{soil\_id}-1)\cdot N_{lc} + \text{lc\_id}$$

Each cell that has both a soil group and a land-cover class gets a unique URH id. (Cells missing either are left 0.)

In [ ]:
N_LC = 8
lc  = rasterio.open(os.path.join(PROC,"landcover_hydro_30m.tif")).read(1)
soil= rasterio.open(os.path.join(PROC,"soils_hydro_30m.tif")).read(1)
valid = (lc>0) & (soil>0)
urh = np.zeros_like(lc, dtype="uint16")
urh[valid] = (soil[valid].astype("uint16")-1)*N_LC + lc[valid]
with rasterio.open(DEM_TIF) as dem:
    up = dem.profile; up.update(dtype="uint16", nodata=0, count=1)
    with rasterio.open(os.path.join(PROC,"urh_30m.tif"),"w",**up) as o: o.write(urh,1)
n = len(np.unique(urh[urh>0]))
print("URH raster written -> urh_30m.tif  |  distinct URH present:", n, "(max possible", 9*N_LC, ")")


## 4. What's done, and what's next

**Done (reusable, extent-independent):**
- `landcover_hydro_30m.tif` — land cover reclassified to 8 hydrological classes, on the DEM grid.
- `soils_hydro_30m.tif` — soils grouped into 9 landscape/hydro groups, on the DEM grid.
- `urh_30m.tif` — the crossed URH map.

**Next (needs the minibacias, i.e. the DEM step / IPH):**
- **Per-minibacia URH composition** — the vector of area fractions `f_{m,u}` per minibacia (notebook 02, §5). It needs
  the minibacia polygons, which come from IPH-HydroTools once the domain/DEM is fixed.

**Refinements to flag for the advisor:**
- Replace the landscape-proxy soil grouping with a **texture / Hydrologic-Soil-Group** classification (from the IGAC
  taxonomy fields or SoilGrids) for proper infiltration behaviour and MUSLE erodibility K.
- Use **period-matched land cover** (IDEAM Corine ~2010–2012 / ESA CCI) for the actual 2011 vs 2015–2016 runs;
  WorldCover 2021 is fine for building the workflow.